In [13]:
## first create image dir of only healthy eyes, (use healthy_annotations.csv)
import sys
import importlib
sys.path.append('/content/drive/MyDrive/Colab Notebooks/')
import preprocessing
importlib.reload(preprocessing)
from preprocessing import preprocessing
import RetinaDataset
importlib.reload(RetinaDataset)
from RetinaDataset import RetinaDataset
import pandas as pd



In [14]:
## first only train on healthy individuals to see if it works
healthy_annotations_path = '/content/drive/MyDrive/Colab Notebooks/healthy_annotations.csv'
og_img_dir = '/content/drive/MyDrive/Colab Notebooks/Images'
processed_img_dir = '/content/drive/MyDrive/Colab Notebooks/preprocessed'


In [15]:
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, ConcatDataset, random_split
import torch

torch.manual_seed(42)  # For reproducibility

def z_normalize(image):
    # Assuming image is a PyTorch tensor of shape (C, H, W)
    # Create a mask for non-black pixels
    mask = image > 0
    for c in range(3):  # Loop over channels
        channel = image[c]
        # Mask out black pixels
        valid_pixels = channel[mask[c]]

        # Calculate mean and std of non-black pixels
        mean = valid_pixels.mean()
        std = valid_pixels.std()

        # Normalize the non-black pixels
        image[c] = torch.where(mask[c], (channel - mean) / (std + 1e-5), channel)
    return image

class TransformedRetinaDataset(RetinaDataset):
    def __init__(self, annotations_file, img_dir, transform=None):
        super().__init__(annotations_file, img_dir)
        self.transform = transform

    def __getitem__(self, idx):
        # Get the sample using the parent class method
        sample = super().__getitem__(idx)
        image, age = sample['image'], sample['age']

        # print("This is type of image after getting it from RetinaDataset:_ ", type(image))

        # Apply the transformations
        if self.transform:
            image = self.transform(image)

        # After other transformations, apply z-normalization
        image = z_normalize(image)


        return {'image': image, 'age': age}

original = transforms.Compose([
    transforms.ToTensor()
])


translate_transform1 = transforms.Compose([
    # transforms.ToPILImage(),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
])

color_jitter_transform1 = transforms.Compose([
    # transforms.ToPILImage(),data/Images/4676_left.jpg
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
])

translate_transform2 = transforms.Compose([
    # transforms.ToPILImage(),
    transforms.RandomAffine(degrees=0, translate=(0.2, 0.2)),
    transforms.ToTensor(),
])

color_jitter_transform2 = transforms.Compose([
    # transforms.ToPILImage(),
    transforms.ColorJitter(brightness=0.6, contrast=0.6),
    transforms.ToTensor(),
])

translate_transform3 = transforms.Compose([
    # transforms.ToPILImage(),
    transforms.RandomAffine(degrees=0, translate=(0.4, 0.4)),
    transforms.ToTensor(),
])

color_jitter_transform3 = transforms.Compose([
    # transforms.ToPILImage(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4),
    transforms.ToTensor(),
])

'''
rotate_transform1 = transforms.Compose([
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
])

rotate_transform2 = transforms.Compose([
    transforms.RandomRotation(degrees=30),
    transforms.ToTensor(),
])

rotate_transform3 = transforms.Compose([
    transforms.RandomRotation(degrees=45),
    transforms.ToTensor(),
])
scale_transform = transforms.Compose([
    transforms.RandomResizedCrop(size=(224, 224), scale=(0.8, 1.2)),
    transforms.ToTensor(),
])
blur_transform = transforms.Compose([
    transforms.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 5)),
    transforms.ToTensor(),
])
'''

# Create the original dataset
# original_dataset = RetinaDataset(annotations_file=healthy_annotations_path, img_dir=processed_img_dir)

# Create augmented datasets with different transformations
datasets = []
# transformations = [original, rotate_transform, translate_transform, flip_and_color_jitter_transform]
transformations = [original,translate_transform1 ,
                   color_jitter_transform1,
                   translate_transform2,
                   color_jitter_transform2,
                   translate_transform3,
                   color_jitter_transform3,
                   # rotate_transform1,
                   # rotate_transform2,
                   # rotate_transform3,
                   # scale_transform,
                   #blur_transform
                   ]


for transform in transformations:
    datasets.append(TransformedRetinaDataset(annotations_file=healthy_annotations_path, img_dir=processed_img_dir, transform=transform))

# Concatenate all datasets
combined_dataset = ConcatDataset(datasets)


# Calculate the lengths of each split
train_length = int(0.8 * len(combined_dataset))
val_length = int(0.1 * len(combined_dataset))
test_length = len(combined_dataset) - train_length - val_length

# Perform the split
train_dataset, val_test_dataset = random_split(combined_dataset, [train_length, val_length + test_length])
val_dataset, test_dataset = random_split(val_test_dataset, [val_length, test_length])

# Create DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [11]:
### trainings and evaluation code ###
import torch
import torch.nn as nn


### trainings and evaluation code ###


def train_model(model, train_loader, val_loader, criterion, optimizer,scheduler,  num_epochs=10, early_stopping_patience=10):
    best_val_loss = float('inf')
    epochs_no_improve = 0
    for epoch in range(num_epochs):
        # training loop
        model.train()
        train_loss = 0.0
        train_mae = 0.0

        for batch in train_loader:
            inputs = batch['image'].to(device)
            labels = batch['age'].to(device)

            optimizer.zero_grad()  # Zero the parameter gradients
            outputs = model(inputs)
            loss = criterion(outputs, labels.unsqueeze(1))
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_mae += torch.abs(outputs - labels.unsqueeze(1)).mean().item()

        train_loss = train_loss / len(train_loader)
        train_mae = train_mae / len(train_loader)

        # validation loop
        model.eval()
        val_loss = 0.0
        val_mae = 0.0
        with torch.no_grad():
            for batch in val_loader:
                inputs = batch['image'].to(device)
                labels = batch['age'].to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels.unsqueeze(1))

                val_loss += loss.item()
                val_mae += torch.abs(outputs - labels.unsqueeze(1)).mean().item()

        val_loss = val_loss / len(val_loader)
        val_mae = val_mae / len(val_loader)
        print(f'Epoch {epoch+1}/{num_epochs} - Training loss: {train_loss:.4f}, Training MAE: {train_mae:.4f}', ', ',  f'Validation loss: {val_loss:.4f}, Validation MAE: {val_mae:.4f}')

         # Early Stopping
        if early_stopping_patience > 0:

          if val_loss < best_val_loss:
              best_val_loss = val_loss
              epochs_no_improve = 0
          else:
            epochs_no_improve += 1
            if epochs_no_improve == early_stopping_patience:
                print("Early stopping triggered")
                break

        # learning rate sheduler
        scheduler.step(val_loss)


### eval on test set
def test_model(model, test_loader, criterion):
    model.eval()

    test_loss = 0.0
    test_mae = 0.0
    with torch.no_grad():
        for batch in test_loader:
            inputs = batch['image'].to(device)
            labels = batch['age'].to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels.unsqueeze(1))

            test_loss += loss.item()
            test_mae += torch.abs(outputs - labels.unsqueeze(1)).mean().item()

    test_loss /= len(test_loader)
    test_mae /= len(test_loader)

    print(f'Test Loss: {test_loss:.4f}, Test MAE: {test_mae:.4f}')




In [12]:
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.models import resnet101, ResNet101_Weights
from torchvision.models import resnet152, ResNet152_Weights
from torchvision import models
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch.optim as optim

### model training and eval ###

##  load pretrained ResNet model with the initialized weights
# model = models.resnet50(weights=ResNet50_Weights.DEFAULT)
# model = models.resnet101(weights=ResNet101_Weights.DEFAULT)
model = models.resnet152(weights=ResNet152_Weights.DEFAULT)


# Freeze all layers initially
for param in model.parameters():
    param.requires_grad = False

# Replace the final fully connected layer
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 1)  # modify it to make a regression at the last layer (age predicition)

# use gpu if possible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print('Working on: ', device)

criterion = nn.MSELoss()  # Mean Squared Error Loss
# criterion = nn.SmoothL1Loss() # used in the similar study
optimizer = optim.Adam(model.parameters(), lr = 1e-3, betas=(0.9, 0.999))
scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.1, patience=5, verbose=True) # reduce lr when validation loss start to plateau

train_model(model, train_loader, val_loader, criterion, optimizer,scheduler=scheduler, num_epochs=45)
test_model(model, test_loader, criterion)

##  Unfreeze all layers and train again:
for param in model.parameters():
    param.requires_grad = True

# Define a new optimizer that includes all parameters (potentially with different learning rates)
optimizer = optim.Adam([
    {'params': model.fc.parameters()},  # Optionally use a higher lr for the newly unfrozen layers
    {'params': model.layer4.parameters(), 'lr': 1e-4},
], lr=1e-4)

train_model(model, train_loader, val_loader, criterion, optimizer, scheduler=scheduler,num_epochs=200, early_stopping_patience=20)
test_model(model,test_loader, criterion)
model_path = 'resnet152_2.pth'
torch.save(model.state_dict(), model_path)

Downloading: "https://download.pytorch.org/models/resnet152-f82ba261.pth" to /root/.cache/torch/hub/checkpoints/resnet152-f82ba261.pth
100%|██████████| 230M/230M [00:01<00:00, 193MB/s]


Working on:  cuda
Epoch 1/100 - Training loss: 1108.6387, Training MAE: 28.3969 ,  Validation loss: 470.4440, Validation MAE: 17.9272
Epoch 2/100 - Training loss: 311.3575, Training MAE: 14.1491 ,  Validation loss: 380.1538, Validation MAE: 15.7369
Epoch 3/100 - Training loss: 277.8762, Training MAE: 13.2765 ,  Validation loss: 300.2923, Validation MAE: 13.9300
Epoch 4/100 - Training loss: 255.0003, Training MAE: 12.6891 ,  Validation loss: 290.3035, Validation MAE: 13.7150
Epoch 5/100 - Training loss: 240.5371, Training MAE: 12.3359 ,  Validation loss: 297.3673, Validation MAE: 14.0523
Epoch 6/100 - Training loss: 224.9947, Training MAE: 11.9241 ,  Validation loss: 244.7452, Validation MAE: 12.5793
Epoch 7/100 - Training loss: 212.1271, Training MAE: 11.5481 ,  Validation loss: 233.6744, Validation MAE: 12.2341
Epoch 8/100 - Training loss: 200.6561, Training MAE: 11.2128 ,  Validation loss: 226.6143, Validation MAE: 12.1529
Epoch 9/100 - Training loss: 195.0487, Training MAE: 11.0189 